# 📝 Text Embedding Extraction — MentalRoBERTa
Reads transcripts from Drive → extracts 768-d embeddings → saves to Drive
**Run Cell 0 → 1 → 2 in order**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Setup
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

DRIVE_DIR       = '/content/drive/MyDrive/edaic'
TRANSCRIPT_DIR  = f'{DRIVE_DIR}/edaic_transcripts'   # ← folder you uploaded
LABELS_CSV      = f'{DRIVE_DIR}/facial_data/labels.csv'
SAVE_DIR        = f'{DRIVE_DIR}/fusion_inputs'
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(SAVE_DIR, exist_ok=True)

# Check transcripts folder exists
if not os.path.exists(TRANSCRIPT_DIR):
    raise FileNotFoundError(f'Upload your transcripts folder to Drive first!\nExpected: {TRANSCRIPT_DIR}')

available = os.listdir(TRANSCRIPT_DIR)
print(f'✅ Found {len(available)} transcript files in Drive')
print(f'   Device: {DEVICE}')
print(f'   Example files: {available[:3]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Read & Clean Transcripts
# ═══════════════════════════════════════════════════════════════
labels_df = pd.read_csv(LABELS_CSV)
processed_texts, pids = [], []
missing, errors = [], []

print(f'Processing {len(labels_df)} participants...')

for pid in labels_df['Participant_ID']:
    path = f'{TRANSCRIPT_DIR}/{pid}_TRANSCRIPT.csv'

    if not os.path.exists(path):
        missing.append(pid)
        continue

    try:
        # Try tab-separated first (E-DAIC standard)
        df = pd.read_csv(path, sep='\t')
        if len(df.columns) < 2:
            df = pd.read_csv(path)   # Fall back to comma

        # Normalise column names
        df.columns = [c.strip().lower() for c in df.columns]

        # Handle 'text' vs 'value' column name
        if 'text' in df.columns and 'value' not in df.columns:
            df['value'] = df['text']

        if 'value' not in df.columns:
            errors.append(f'{pid}: no text column (found: {df.columns.tolist()})')
            continue

        # Filter participant turns if speaker column exists
        if 'speaker' in df.columns:
            mask = df['speaker'].astype(str).str.strip().str.lower() == 'participant'
            text_parts = df[mask]['value'].astype(str).tolist()
        else:
            # No speaker column — whole file is participant speech
            text_parts = df['value'].astype(str).tolist()

        full_text = ' '.join(text_parts).strip()
        if not full_text:
            errors.append(f'{pid}: empty text')
            continue

        processed_texts.append(full_text)
        pids.append(pid)

    except Exception as e:
        errors.append(f'{pid}: {e}')

print(f'\n✅ Ready  : {len(processed_texts)} participants')
print(f'⚠️  Missing: {len(missing)} transcripts')
print(f'❌ Errors : {len(errors)}')
if errors[:3]:
    print('   First errors:', errors[:3])

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Extract 768-d Embeddings with MentalRoBERTa
# ═══════════════════════════════════════════════════════════════
MODEL_NAME = 'mental/mental-roberta-base'
print(f'Loading {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print(f'✅ Model loaded on {DEVICE}')

embeddings = []

print(f'\nExtracting embeddings for {len(processed_texts)} participants...')
with torch.no_grad():
    for text in tqdm(processed_texts):
        enc = tokenizer(
            text,
            return_tensors='pt',
            max_length=512,
            truncation=True,
            padding='max_length'
        ).to(DEVICE)

        out  = model(**enc)

        # Mean pooling over all tokens (better than [CLS] for long text)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        embeddings.append(emb.squeeze().cpu().numpy())

emb_arr = np.array(embeddings)
pid_arr = np.array(pids)

# Save
np.save(f'{SAVE_DIR}/text_embeddings_all.npy',  emb_arr)
np.save(f'{SAVE_DIR}/participant_ids_text.npy', pid_arr)

print(f'\n✅ SUCCESS!')
print(f'   Shape   : {emb_arr.shape}     ← should be (~275, 768)')
print(f'   Std dev : {emb_arr.std():.4f}  ← should be > 0.05')
print(f'   Saved to: {SAVE_DIR}')
print(f'\n🚀 Now run the multimodal_fusion.ipynb notebook!')